# Projekt 01 (basic): SQL — Daten holen, wo sie wohnen

**Ziel:** Die SQL-Grundbausteine (SELECT, WHERE, GROUP BY, JOIN, HAVING) an einer
richtigen Datenbank ueben — und nebenbei sehen, wie SQL und pandas zusammenspielen.

**Vorbereitung** (einmalig, im Ordner `01-basic`, venv aktiv):

```
python generate_db.py
```

Das erzeugt `daten/shop.db`: eine SQLite-Datenbank mit einem fiktiven Online-Shop —
**60 Kunden, 25 Produkte, 800 Bestellungen** in drei verknuepften Tabellen:

```
kunden(kunden_id, name, stadt, registriert_am)
produkte(produkt_id, name, kategorie, preis)
bestellungen(bestell_id, kunden_id -> kunden, produkt_id -> produkte, menge, bestellt_am)
```

Beachte das Prinzip: Die Bestellung speichert nur *Verweise* (Fremdschluessel) auf
Kunde und Produkt — Namen und Preise stehen genau einmal in ihren eigenen Tabellen
(Normalisierung). Um "Umsatz pro Stadt" zu beantworten, muss man die Tabellen
**joinen** — genau das uebst du hier.

**Bezug zum Skript:** Abschnitt 2.4 (SQL). Tipp vorab: 15 Minuten https://sqlbolt.com.

## 1. Verbindung aufbauen

SQLite ist in Python eingebaut; `pd.read_sql` fuehrt eine Abfrage aus und liefert
das Ergebnis direkt als DataFrame. Die Hilfsfunktion `q(...)` spart Tipparbeit.

In [ ]:
import os
import sqlite3
import pandas as pd

PFAD = "daten/shop.db" if os.path.exists("daten/shop.db") else "../daten/shop.db"
con = sqlite3.connect(PFAD)

def q(sql):
    """Fuehrt eine SQL-Abfrage aus und liefert einen DataFrame."""
    return pd.read_sql(sql, con)

q("SELECT name FROM sqlite_master WHERE type = 'table'")

## 2. SELECT, WHERE, ORDER BY, LIMIT

Erstes Muster: Zeilen auswaehlen, filtern, sortieren, begrenzen.

```sql
SELECT spalte1, spalte2 FROM tabelle WHERE bedingung ORDER BY spalte DESC LIMIT 5
```

**Aufgaben:**
1. Alle Produkte der Kategorie 'Buecher' — Name und Preis, nach Preis absteigend.
2. Wie viele Produkte kosten mehr als 50 EUR? (`COUNT(*)`)
3. Die 5 juengsten Bestellungen (nach `bestellt_am`).

In [ ]:
# Aufgabe 1:
# TODO: q("SELECT ... FROM produkte WHERE ... ORDER BY ...")

In [ ]:
# Aufgaben 2 und 3:
teuer = ...     # TODO
# TODO: neueste = ...
print(neueste)

# Mini-Check:
print(int(teuer["anzahl"][0]) == 6)

## 3. GROUP BY — Aggregieren wie groupby

**Aufgaben:**
1. Wie viele Produkte gibt es pro Kategorie? (`COUNT(*)`, `GROUP BY`)
2. Durchschnittspreis pro Kategorie, absteigend sortiert (`AVG`, `ROUND(x, 2)`).
3. Bestellungen pro Monat: `strftime('%m', bestellt_am)` extrahiert den Monat —
   gibt es einen Dezember-Peak?

In [ ]:
# TODO: drei Abfragen wie in den Aufgaben beschrieben
# TODO: monate = q(...)
monate.plot.bar(x="monat", y="anzahl", legend=False, title="Bestellungen pro Monat");

## 4. JOIN — Tabellen verknuepfen

Der eigentliche Grund, warum die Daten in drei Tabellen liegen. Muster:

```sql
SELECT ... FROM bestellungen b
JOIN kunden   k ON k.kunden_id  = b.kunden_id
JOIN produkte p ON p.produkt_id = b.produkt_id
```

**Aufgaben:**
1. Anzahl Bestellungen pro **Stadt** (braucht: bestellungen ⋈ kunden).
2. **Umsatz** (= `p.preis * b.menge`) pro **Kategorie**, absteigend.
3. Die **Top-3-Kunden** nach Umsatz (Name + Umsatz; braucht alle drei Tabellen!).

In [ ]:
# TODO: drei JOIN-Abfragen wie beschrieben
# TODO: umsatz_kat = q(...); top3 = q(...)
print(top3)

# Mini-Checks:
print(umsatz_kat["kategorie"][0] == "Elektronik")          # umsatzstaerkste Kategorie
print(top3["name"][0] == "Emma Schulz")                    # Top-Kundin

## 5. LEFT JOIN — wer fehlt?

Ein `JOIN` (INNER) zeigt nur Kunden, die auch bestellt haben. Marketing fragt aber:
**"Welche Kunden haben noch NIE bestellt?"** Dafuer: `LEFT JOIN` (alle Kunden behalten,
Bestellspalten sind dann `NULL`) plus `WHERE b.bestell_id IS NULL`.

**Aufgabe:** Finde diese Kunden (Name, Stadt). Es sollten **5** sein.

In [ ]:
karteileichen = q(...)   # TODO: LEFT JOIN + IS NULL
print(karteileichen)
print(len(karteileichen) == 5)

## 6. WHERE vs. HAVING

`WHERE` filtert Zeilen **vor** der Gruppierung, `HAVING` filtert Gruppen **danach** —
nur dort darf `COUNT(*)` & Co. stehen (Skript 2.4, Selbsttest-Frage 7!).

**Aufgabe:** Welche Staedte haben 150 oder mehr Bestellungen? (Erwartet: 4 Staedte.)
Probiere zuerst absichtlich `WHERE COUNT(*) >= 150` — lies die Fehlermeldung.

In [ ]:
staedte = q(...)   # TODO: GROUP BY + HAVING
print(staedte)
print(len(staedte) == 4)

## 7. Arbeitsteilung SQL + pandas

Faustregel aus dem Skript: **Filtern/Joinen in SQL, Analysieren/Plotten in pandas.**
So sieht der typische Workflow aus — SQL liefert die kompakte Analysetabelle,
pandas macht den Rest (fertig vorgegeben):

In [ ]:
analyse = q("""SELECT b.bestellt_am, k.stadt, p.kategorie,
                      p.preis * b.menge AS umsatz
               FROM bestellungen b
               JOIN kunden   k ON k.kunden_id  = b.kunden_id
               JOIN produkte p ON p.produkt_id = b.produkt_id""")
analyse["bestellt_am"] = pd.to_datetime(analyse["bestellt_am"])

pivot = analyse.pivot_table(index="stadt", columns="kategorie",
                            values="umsatz", aggfunc="sum").round(0)
print(pivot)
pivot.plot.bar(stacked=True, figsize=(8, 4), title="Umsatz pro Stadt und Kategorie");

In [ ]:
con.close()   # Verbindung am Ende immer schliessen

## Geschafft — was du jetzt kannst

- SELECT / WHERE / ORDER BY / LIMIT, Aggregation mit GROUP BY
- Tabellen ueber Fremdschluessel joinen (INNER und LEFT) — und wissen, wann welcher
- WHERE vs. HAVING sicher unterscheiden
- SQL-Ergebnisse nahtlos in pandas weiterverarbeiten

**Bonusaufgaben** (optional):
1. Umsatz pro Monat als SQL-Abfrage (strftime + JOIN + SUM) — und als Linienplot.
2. Welcher Kunde hat die meisten *verschiedenen* Produkte gekauft? (`COUNT(DISTINCT ...)`)
3. Schreibe die Stadt-Bestellungen-Abfrage aus Abschnitt 4 einmal komplett in pandas
   nach (merge + groupby) und vergleiche die Ergebnisse.